# **ToeplitzResnet Model Training for Toeplitz Matrix Interpolation**

This notebook implements the training pipeline for the ToeplitzResNet model, designed to perform Toeplitz covariance matrix interpolation in sparse array signal processing applications. The objective is to reconstruct a complete Toeplitz covariance matrix from partially observed or corrupted measurements, which is essential for improving the performance of downstream tasks such as Direction-of-Arrival (DoA) estimation.

The model leverages a Residual Neural Network (ResNet) architecture to learn the mapping between incomplete covariance matrices and their corresponding full Toeplitz structures. By exploiting the inherent Toeplitz property of covariance matrices in uniform linear arrays, the network learns to accurately recover missing elements while maintaining structural consistency.

## **Import Libraries and Modules**
- File Operations and Environments
- Data Preparation Modules
- Deep Learning Library
- Model Training and Evaluation Modules

In [ ]:
import sys

root_dir = '/home/iot/Sajid/Repositories/DL-DoA'
print(f"Root directory: {root_dir}")
sys.path.insert(0, root_dir)

In [ ]:
# Data preparation
import os
import numpy as np
from data.processor import (
    get_dataloaders,
    complex_to_tensor,
    tensor_to_complex
)
from torch.utils.data import DataLoader

# Deep learning
import torch
from torch import nn

# Model training and evaluation
from models.cnn_models import ToeplitzResNet
from models.losses import PINNLoss
from models.utils import Trainer, model_inference, evaluate_models
import matplotlib.pyplot as plt

## **Load And Process Data**

Load simulated data from the directory and process for model training.

In [ ]:
data_dir = '/home/iot/Sajid/Repositories/DL-DoA/data/CATARACS_M6_N7_DOF83'

train_data, val_data, test_data = get_dataloaders(
    obs_dir=os.path.join(data_dir, 'obs'),
    true_dir=os.path.join(data_dir, 'true'),
    val_size=0.1,
    test_size=0.1,
    seed=42
)

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Testing samples: {len(test_data)}")

In [ ]:
# Make DataLoader for training
batch_size = 64

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

print(f"Number of batches in training: {len(train_loader)}")
print(f"Number of batches in validation: {len(val_dataloader)}")
print(f"Number of batches in testing: {len(test_dataloader)}")

## **Initialize Models**

In [ ]:
model = ToeplitzResNet(num_blocks=8, input_channels=2)
mse_loss = nn.MSELoss()
pinn_loss = PINNLoss(lambda_toeplitz=0.5, lambda_hermitian=0.5)

# Setup optimizer
lr = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

print("Model Structure:")
model

## **Training with MSELoss**

In [ ]:
# Initialize trainer
trainer_mse = Trainer(
    model=model,
    criterion=mse_loss,
    optimizer=optimizer,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    save_path=os.path.join(root_dir, 'saved_models'),
    file_name='toeplitz_resnet_mse.pth',
    patience=5
)

In [ ]:
# Train the model
num_epochs = 50
history_mse = trainer_mse.fit(
    train_loader=train_loader,
    val_loader=val_dataloader,
    epochs=num_epochs
)

In [ ]:
# Plot training history
plt.figure(figsize=(12, 5))
plt.plot(history_mse['train_loss'], label='Train Loss')
plt.plot(history_mse['val_loss'], label='Val Loss')
plt.title('MSE Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
# Evaluate on test set
test_metrics_mse = evaluate_models(
    model=model,
    dataloader=test_dataloader,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
print("Test Metrics (MSE):", test_metrics_mse)

## **Training with PINNLoss**

In [ ]:
trainer_pinn = Trainer(
    model=model,
    criterion=pinn_loss,
    optimizer=optimizer,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    save_path=os.path.join(root_dir, 'saved_models'),
    file_name='toeplitz_resnet_pinn.pth',
    patience=5
)

In [ ]:
# Train the model with PINN loss
num_epochs = 50
history_pinn = trainer_pinn.fit(
    train_loader=train_loader,
    val_loader=val_dataloader,
    epochs=num_epochs
)

In [ ]:
# Plot training history with PINN loss
plt.figure(figsize=(12, 5))
plt.plot(history_pinn['train_loss'], label='Train Loss')
plt.plot(history_pinn['val_loss'], label='Val Loss')
plt.title('PINN Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
# Evaluate on test set for PINN-trained model
test_metrics_pinn = evaluate_models(
    model=model,
    dataloader=test_dataloader,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
print("Test Metrics (PINN):", test_metrics_pinn)